# 03 — Analyze Website Visitor Locations from Apache/Nginx Logs with IP2Location.io

**Use case:** Extract client IP addresses from standard Apache/Nginx access logs, enrich the **unique public IPs** through the IP2Location.io API, and build country-level website traffic reports and charts.

**You will learn to:**
1. Read Apache/Nginx **Combined** and **Common** access-log records.
2. Extract IPs, timestamps, URL paths, HTTP status codes, and response bytes.
3. Validate/deduplicate global IPv4 and IPv6 addresses **before** calling the API.
4. Query `https://api.ip2location.io/` with optional Bearer authentication and defensive error handling.
5. Analyze requests by country and HTTP status; plot the top countries.
6. Export an enriched CSV for subsequent analysis.

**Download alongside this notebook:** `sample_access.log`. If it is missing, the notebook creates a small synthetic demo log without overwriting existing files. **The example public IPs belong to DNS services; they are synthetic placeholders, not actual website visitors.** Geolocation may not represent human visitors.

API documentation: [IP2Location.io IP Geolocation API](https://www.ip2location.io/ip2location-documentation).

## 1. Install dependencies

Run this cell inside Jupyter. The standard-library modules used later do not need installation.

In [ ]:
%pip install -q requests pandas matplotlib

## 2. Configure paths and query limits

- Place an Apache/Nginx log beside this notebook, or change `INPUT_LOG` to its full path.
- Leave `IP2LOCATION_API_KEY` unset for **keyless** requests. For authenticated queries, set it as an environment variable **before launching Jupyter** (do not paste secrets into shared notebooks).
- `MAX_UNIQUE_LOOKUPS` prevents unexpectedly costly queries on large production log files. Only that many distinct, public IPs will be queried.
- Set `RUN_API_LOOKUPS=False` to test parsing and reporting **offline**; unmatched rows then receive a non-success status and are excluded from geographic summaries.

The standard endpoint performs **one request per unique IP**. Keyless access currently has a **1,000 queries/day** limit. A registered Free Plan currently offers **50,000 queries/month**; verify current limits and required attribution on the [official pricing page](https://www.ip2location.io/pricing). The bulk API is a separate paid feature.

In [ ]:
import ipaddress
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display

API_URL = "https://api.ip2location.io/"
API_KEY = os.environ.get("IP2LOCATION_API_KEY", "").strip()
INPUT_LOG = Path("sample_access.log")
OUTPUT_CSV = Path("website_visitors_geolocated.csv")
RUN_API_LOOKUPS = True
MAX_UNIQUE_LOOKUPS = 20
REQUEST_DELAY_SECONDS = 0.25
MAX_RETRIES = 1

assert MAX_UNIQUE_LOOKUPS >= 0 and REQUEST_DELAY_SECONDS >= 0 and MAX_RETRIES >= 0
print(f"Input log: {INPUT_LOG} | Output CSV: {OUTPUT_CSV}")
print(f"API mode: {'Bearer token' if API_KEY else 'keyless'} | Execute API: {RUN_API_LOOKUPS}")

## 3. Prepare a synthetic sample log, if needed

The separate download already contains a demo log with IPv4/IPv6, public and private IPs, repeat visits, HTTP errors, one Common-format entry, and one malformed line. If no input file is present, the cell generates equivalent examples. It never overwrites an existing log.

**Security note:** Real access logs can contain personal data and query strings. This notebook extracts request paths and sends only distinct public IPs to IP2Location.io. Review your privacy obligations and log-retention policy before using real traffic.

In [ ]:
if not INPUT_LOG.exists():
    demo_ips = ["8.8.8.8", "1.1.1.1", "9.9.9.9", "208.67.222.222",
                "2606:4700:4700::1111", "192.168.1.20", "8.8.8.8"]
    demo_paths = ["/", "/pricing", "/docs", "/blog", "/contact", "/health", "/"]
    from datetime import timedelta
    start = datetime(2026, 10, 26, 8, 0, tzinfo=timezone.utc)
    demo_lines = []
    for i, (ip, path) in enumerate(zip(demo_ips, demo_paths)):
        stamp = (start + timedelta(hours=i)).strftime("%d/%b/%Y:%H:%M:%S %z")
        demo_lines.append(
            f'{ip} - - [{stamp}] "GET {path} HTTP/1.1" 200 {500 + i * 25} "-" "DemoBrowser/1.0"'
        )
    demo_lines.append("malformed demo record")
    INPUT_LOG.write_text("\n".join(demo_lines) + "\n", encoding="utf-8")
    print(f"Created a synthetic example: {INPUT_LOG}")
else:
    print(f"Using existing log file: {INPUT_LOG}")

## 4. Parse Apache/Nginx Common and Combined log lines

A standard access-log line begins with the client IP, followed by identity fields, a timestamp, a quoted request, an HTTP status, and byte count. Combined format adds quoted referrer and user-agent fields. This parser intentionally supports these **standard formats**, not arbitrary custom log formats, JSON logs, or rotated `.gz` files.

**Behind a reverse proxy/CDN:** The first field can be the proxy IP. Configure Apache `mod_remoteip` or Nginx `real_ip` to record the true client IP **only from trusted proxy addresses**. Do not automatically trust client-supplied `X-Forwarded-For` headers.

In [ ]:
ACCESS_LOG_RE = re.compile(
    r'^(?P<ip>\S+) \S+ \S+ \[(?P<timestamp>[^\]]+)\] '
    r'"(?P<request>(?:\\.|[^"\\])*)" '
    r'(?P<http_status>\d{3}) (?P<response_bytes>\d+|-)'
    r'(?: "(?:\\.|[^"\\])*" "(?:\\.|[^"\\])*")?\s*$'
)


def parse_access_log_line(line: str, line_number: int) -> dict | None:
    """Return a normalized record for a Common/Combined log line, or None."""
    match = ACCESS_LOG_RE.match(line.rstrip("\r\n"))
    if not match:
        return None
    item = match.groupdict()
    try:
        timestamp = datetime.strptime(item["timestamp"], "%d/%b/%Y:%H:%M:%S %z")
    except ValueError:
        return None
    request_parts = item["request"].split(" ", 2)
    method = request_parts[0] if request_parts else ""
    request_target = request_parts[1] if len(request_parts) > 1 else ""
    # Remove query strings/fragments to avoid retaining possible sensitive parameters.
    path = request_target.split("?", 1)[0].split("#", 1)[0]
    return {
        "line_number": line_number,
        "ip": item["ip"],
        "timestamp_utc": timestamp.astimezone(timezone.utc),
        "method": method,
        "path": path,
        "http_status": int(item["http_status"]),
        "response_bytes": 0 if item["response_bytes"] == "-" else int(item["response_bytes"]),
    }


parsed_rows = []
bad_lines = 0
with INPUT_LOG.open("r", encoding="utf-8", errors="replace") as handle:
    for line_number, line in enumerate(handle, start=1):
        record = parse_access_log_line(line, line_number)
        if record is None:
            bad_lines += 1
        else:
            parsed_rows.append(record)

if not parsed_rows:
    raise ValueError("No supported Apache/Nginx Common/Combined log entries were found.")

logs_df = pd.DataFrame(parsed_rows)
print(f"Parsed {len(logs_df):,} log entries; skipped {bad_lines:,} malformed/unsupported entries")
display(logs_df.head(10))

## 5. Validate IP addresses and count planned lookups

Use Python's `ipaddress` module to normalize IPv4 and IPv6. Only addresses for which `.is_global` is `True` are queried; loopback, local/private, reserved and malformed addresses are excluded. Each unique public IP costs at most one API query during this run, regardless of how many requests it made.

In [ ]:
def prepare_ip(raw_value: str) -> tuple[str, str]:
    try:
        parsed = ipaddress.ip_address(raw_value.strip())
    except ValueError:
        return "", "invalid_ip"
    if not parsed.is_global:
        return str(parsed), "non_public_ip"
    return str(parsed), "ready"

ip_prepared = logs_df["ip"].map(prepare_ip)
logs_df["normalized_ip"] = ip_prepared.map(lambda pair: pair[0])
logs_df["ip_status"] = ip_prepared.map(lambda pair: pair[1])
unique_public_ips = list(dict.fromkeys(
    logs_df.loc[logs_df["ip_status"] == "ready", "normalized_ip"]
))
lookup_ips = unique_public_ips[:MAX_UNIQUE_LOOKUPS]
print(f"Total log requests: {len(logs_df):,}")
print(f"Distinct public IPs: {len(unique_public_ips):,}")
print(f"Planned API calls: {len(lookup_ips) if RUN_API_LOOKUPS else 0:,}")
print(f"Unqueried due to cap: {len(unique_public_ips) - len(lookup_ips):,}")
print("IP validation statuses:")
print(logs_df["ip_status"].value_counts().to_string())

## 6. Look up unique public IPs with IP2Location.io

The request uses the JSON endpoint, an optional `Authorization: Bearer` header, a timeout, and limited retries on temporary failures. It stops further queries when an authentication/quota error occurs (including API error code `10000`). Missing fields are expected: returned attributes depend on the account's plan.

**Note:** This notebook does not save API credentials or query responses to a separate persistent cache. Rerunning the lookup cell may use additional quota.

In [ ]:
GEO_FIELDS = ["country_code", "country_name", "region_name", "city_name",
              "latitude", "longitude", "time_zone", "asn", "as", "is_proxy"]

class QuotaOrAuthError(RuntimeError):
    """Raised when further requests should not be attempted."""


def lookup_ip(session: requests.Session, ip: str) -> dict:
    for attempt in range(MAX_RETRIES + 1):
        try:
            response = session.get(
                API_URL, params={"ip": ip, "format": "json"}, timeout=(5, 12)
            )
        except requests.RequestException as exc:
            if attempt < MAX_RETRIES:
                time.sleep(1 + attempt)
                continue
            return {"lookup_status": "network_error", "lookup_error": str(exc)[:200]}

        if response.status_code in (401, 403, 429):
            raise QuotaOrAuthError(f"HTTP {response.status_code}; check API key or rate limit")
        if response.status_code >= 500 and attempt < MAX_RETRIES:
            time.sleep(1 + attempt)
            continue

        try:
            data = response.json()
        except ValueError:
            return {"lookup_status": "invalid_response", "lookup_error": f"HTTP {response.status_code}: non-JSON body"}

        if not isinstance(data, dict):
            return {"lookup_status": "invalid_response", "lookup_error": "Unexpected JSON shape"}
        error = data.get("error")
        if isinstance(error, dict):
            code = error.get("error_code")
            message = str(error.get("error_message", "Unknown API error"))
            if str(code) == "10000":
                raise QuotaOrAuthError(f"API error {code}: {message}")
            return {"lookup_status": "api_error", "lookup_error": f"{code}: {message}"}
        if not response.ok:
            return {"lookup_status": "http_error", "lookup_error": f"HTTP {response.status_code}"}
        if not data.get("country_code"):
            return {"lookup_status": "invalid_response", "lookup_error": "No country_code in API response"}

        return {
            "lookup_status": "ok", "lookup_error": "",
            **{field: data.get(field) for field in GEO_FIELDS},
        }
    return {"lookup_status": "http_error", "lookup_error": "Retry limit reached"}

## 7. Execute lookups within the safety cap

Make requests only for eligible IPs. Duplicate entries reuse the same lookup. If quota/authentication fails, the remaining lookups are marked as skipped instead of requesting them. In offline mode, the report will still show traffic counts, but geographic metrics will be empty (not fabricated).

In [ ]:
lookup_results = {
    ip: {"lookup_status": "skipped_limit", "lookup_error": "MAX_UNIQUE_LOOKUPS cap"}
    for ip in unique_public_ips[MAX_UNIQUE_LOOKUPS:]
}

if not RUN_API_LOOKUPS:
    for ip in lookup_ips:
        lookup_results[ip] = {"lookup_status": "skipped_offline", "lookup_error": "RUN_API_LOOKUPS=False"}
else:
    with requests.Session() as session:
        if API_KEY:
            session.headers.update({"Authorization": f"Bearer {API_KEY}"})
        for index, ip in enumerate(lookup_ips):
            try:
                lookup_results[ip] = lookup_ip(session, ip)
            except QuotaOrAuthError as exc:
                lookup_results[ip] = {"lookup_status": "quota_or_auth_error", "lookup_error": str(exc)}
                for remaining in lookup_ips[index + 1:]:
                    lookup_results[remaining] = {
                        "lookup_status": "skipped_quota", "lookup_error": "Stopped after an authentication or quota failure"
                    }
                print(f"Stopping lookups: {exc}")
                break
            print(f"[{index + 1}/{len(lookup_ips)}] {ip}: {lookup_results[ip]['lookup_status']}")
            if index + 1 < len(lookup_ips):
                time.sleep(REQUEST_DELAY_SECONDS)

print("Unique-IP lookup result counts:")
print(pd.Series([item["lookup_status"] for item in lookup_results.values()]).value_counts().to_string()
      if lookup_results else "No public IPs to query")

## 8. Enrich every log row and export a CSV

Each original request stays in the dataset. Requests from the same IP share geolocation attributes, while non-public, invalid or unqueried IPs receive explicit statuses and no fabricated location. The export file may contain sensitive request metadata, so keep it private and apply appropriate retention controls.

In [ ]:
enriched_rows = []
for row in logs_df.to_dict(orient="records"):
    if row["ip_status"] == "ready":
        geo = lookup_results.get(row["normalized_ip"], {
            "lookup_status": "not_queried", "lookup_error": "Not processed"
        })
    else:
        geo = {"lookup_status": row["ip_status"], "lookup_error": "IP not globally routable or invalid"}
    enriched_rows.append({
        **row, **{key: geo.get(key) for key in ["lookup_status", "lookup_error", *GEO_FIELDS]}
    })

enriched_df = pd.DataFrame(enriched_rows)
enriched_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved {len(enriched_df):,} rows to {OUTPUT_CSV.resolve()}")
print("Per-request lookup status:")
print(enriched_df["lookup_status"].value_counts().to_string())
display(enriched_df[["timestamp_utc", "ip", "path", "http_status", "country_name", "lookup_status"]].head(10))

## 9. Summarize website traffic by country

Count **HTTP requests (page/resource hits)** by geolocated country, not people or sessions. Unique IP counts are useful context but **are not unique visitors**: NAT, VPNs, proxies, bots, and shared addresses make that assumption inaccurate.

Include only successful lookups in country-level totals; report the unlocated request count separately so incomplete coverage is visible.

In [ ]:
located = enriched_df[enriched_df["lookup_status"] == "ok"].copy()
unlocated_requests = len(enriched_df) - len(located)
print(f"Requests with API geolocation: {len(located):,} / {len(enriched_df):,}")
print(f"Requests without API geolocation: {unlocated_requests:,}")

if located.empty:
    country_summary = pd.DataFrame(columns=["country_name", "requests", "unique_ips", "http_errors", "response_bytes"])
    print("No successful geolocation results yet. Run with API enabled for a country report.")
else:
    located["is_http_error"] = located["http_status"] >= 400
    country_summary = (located.groupby("country_name", dropna=False)
        .agg(requests=("ip", "size"),
             unique_ips=("normalized_ip", "nunique"),
             http_errors=("is_http_error", "sum"),
             response_bytes=("response_bytes", "sum"))
        .reset_index()
        .sort_values(["requests", "country_name"], ascending=[False, True]))
    display(country_summary)

# Non-geographic traffic analysis works even when API access is disabled.
http_status_summary = (enriched_df.groupby("http_status")
                       .size().rename("requests").reset_index()
                       .sort_values("http_status"))
print("HTTP status breakdown (all parsed requests):")
display(http_status_summary)

## 10. Visualize the top traffic countries

This chart counts requests by country **only where geolocation succeeded**. The sample addresses are public DNS endpoints used to illustrate the process, not real users; use your own access logs for meaningful traffic analysis.

In [ ]:
if country_summary.empty:
    print("No country data to chart. Check API access and rerun Sections 7–10.")
else:
    top = country_summary.head(10).iloc[::-1]
    plt.figure(figsize=(9, max(3, len(top) * 0.55)))
    plt.barh(top["country_name"].fillna("Unknown"), top["requests"])
    plt.xlabel("HTTP requests")
    plt.ylabel("Geolocated country")
    plt.title("Top 10 countries by web requests (successfully geolocated only)")
    plt.tight_layout()
    plt.show()

## 11. Find the most frequently requested paths

Path counts work even if API access is unavailable. The parser removes query strings to reduce accidental retention of tokens or personal data in the analysis.

In [ ]:
top_paths = (enriched_df.groupby("path").size().rename("requests")
             .reset_index().sort_values("requests", ascending=False).head(10))
display(top_paths)

# Optional: requests over time in UTC (hourly buckets).
hourly = (enriched_df.set_index("timestamp_utc").resample("1h").size()
          .rename("requests"))
plt.figure(figsize=(10, 3.5))
plt.plot(hourly.index, hourly.values)
plt.title("Request volume by hour (UTC)")
plt.ylabel("HTTP requests")
plt.xlabel("Hour (UTC)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 12. Use your own Apache/Nginx logs

1. Update `INPUT_LOG` to a **standard Common/Combined-format**, UTF-8-compatible access log (for example, `Path("/var/log/nginx/access.log")`). Confirm read permissions.
2. Verify which address is recorded as the client IP, especially if your site uses Cloudflare, a load balancer, or another reverse proxy. Trust forwarded IP headers only from configured, known proxies.
3. Use a small subset of logs first and check `MAX_UNIQUE_LOOKUPS` before making requests. Larger logs may need streaming, a persistent deduplication cache, and explicit quota management.
4. Set the `IP2LOCATION_API_KEY` environment variable if using an API-key plan. Restart Jupyter or rerun the configuration cell, then rerun Sections 4–11.
5. Open `website_visitors_geolocated.csv` in a spreadsheet or data-analysis tool for further reporting.

**Interpreting the report:** Geolocation is approximate; a country reflects where an IP is estimated to be, not a verified user location. Requests include bots and assets, so counts are not pageviews, sessions or humans. Some attributes vary by subscription plan. Consider minimizing or anonymizing retained IPs and protecting the exported file.

**References:** [API documentation](https://www.ip2location.io/ip2location-documentation) · [Pricing and current usage limits](https://www.ip2location.io/pricing).